In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


QUESTION_1:Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

**Introduction to Hugging Face transformers and datasets**

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

train = dataset["train"]

print(train)

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})


In [3]:
def combine(example):
    example["combined_text"] = (
        example["prompt"] + " " +
        example["A"] + " " +
        example["B"] + " " +
        example["C"] + " " +
        example["D"] + " " +
        example["E"]
    )
    return example

train = train.map(combine)

print(train[51]["combined_text"])
print(len(train[51]["combined_text"]))

Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs. Class L dwarfs are cooler than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are dark red in color and are brightest in infrared. Their atmosphere is cool enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore star

QUESTION_2:Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.vocab_size)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


30522


QUESTION_3:Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.sep_token)
print(tokenizer.sep_token_id)

[SEP]
102


QUESTION_4:Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [6]:
print(train)
print(train.features)
print(train[0])
print(train["prompt"][:5])

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'combined_text'],
    num_rows: 2000
})
{'id': Value('int64'), 'prompt': Value('string'), 'A': Value('string'), 'B': Value('string'), 'C': Value('string'), 'D': Value('string'), 'E': Value('string'), 'answer': Value('string'), 'combined_text': Value('string')}
{'id': 1, 'prompt': "Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.", 'A': "Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.", 'B': 'Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationshi

In [7]:
print(train.features)
print(train[0]["prompt"])
print(type(train[0]["prompt"]))

{'id': Value('int64'), 'prompt': Value('string'), 'A': Value('string'), 'B': Value('string'), 'C': Value('string'), 'D': Value('string'), 'E': Value('string'), 'answer': Value('string'), 'combined_text': Value('string')}
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
<class 'str'>


In [8]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

train = dataset["train"]

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Convert to a plain Python list of strings
prompts = list(train["prompt"])

# Tokenize
encodings = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encodings["input_ids"].shape)

torch.Size([2000, 128])


**BERT/RoBERTa Architecture & Attention Mechanisms**

QUESTION_5: A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

In [9]:
hidden_size = 768
num_attention_heads = 12

head_dim = hidden_size // num_attention_heads

print("Hidden Size:", hidden_size)
print("Number of Attention Heads:", num_attention_heads)
print("Dimension of each Attention Head:", head_dim)

Hidden Size: 768
Number of Attention Heads: 12
Dimension of each Attention Head: 64


QUESTION_6:Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here.

In [10]:
from transformers import AutoTokenizer, AutoModel

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Row ID 0
text = train[0]["prompt"]

# Default tokenization (NO padding/truncation)
inputs = tokenizer(text, return_tensors="pt")

# Forward pass
outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

torch.Size([1, 31, 768])


QUESTION_7:  Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  
*


In [11]:
cls_vector = outputs.last_hidden_state[0, 0]

answer = cls_vector[:5].sum().item()

print(round(answer, 4))

-1.2001


QUESTION_8:Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

*


In [12]:
import torch
from transformers import AutoTokenizer, AutoModel

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

sentence = "Light-ion fusion is a technique."

# Tokenize
inputs = tokenizer(sentence, return_tensors="pt")

# Show tokens and their indices
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Tokens:")
for i, token in enumerate(tokens):
    print(i, token)

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# Last layer attention
last_attention = outputs.attentions[-1]   # (batch, heads, seq_len, seq_len)

# First head
head0 = last_attention[0, 0]

# Find index of 'fusion'
fusion_idx = tokens.index("fusion")

# CLS token is index 0
attention_weight = head0[0, fusion_idx].item()

print("\nFusion index:", fusion_idx)
print("Attention Weight:", round(attention_weight, 4))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Tokens:
0 [CLS]
1 light
2 -
3 ion
4 fusion
5 is
6 a
7 technique
8 .
9 [SEP]

Fusion index: 4
Attention Weight: 0.1025


**Context-Aware Embeddings**

QUESTION_9:Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [13]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd

# Load the dataset
train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

# Load the Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Get prompt and Option B for row ID 0
prompt = train.loc[0, "prompt"]
option_b = train.loc[0, "B"]

print("Prompt:", prompt)
print("Option B:", option_b)

# Generate embeddings
prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_embedding = model.encode(option_b, convert_to_tensor=True)

# Cosine similarity
similarity = util.cos_sim(prompt_embedding, option_embedding)

print("Cosine Similarity:", round(similarity.item(), 4))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
Option B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Cosine Similarity: 0.7658


QUESTION_10:Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [14]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util

# Load dataset
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

options = ["A", "B", "C", "D", "E"]


# TF-IDF PIPELINE


combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

tfidf = TfidfVectorizer(stop_words="english")
tfidf.fit(combined_text)

tfidf_top3 = []
tfidf_scores = []

for _, row in train.iterrows():

    prompt_vec = tfidf.transform([row["prompt"]])

    sims = []

    for opt in options:
        opt_vec = tfidf.transform([row[opt]])
        score = cosine_similarity(prompt_vec, opt_vec)[0][0]
        sims.append((opt, score))

    sims.sort(key=lambda x: x[1], reverse=True)

    preds = [x[0] for x in sims[:3]]
    tfidf_top3.append(preds)

    if row["answer"] in preds:
        tfidf_scores.append(1/(preds.index(row["answer"])+1))
    else:
        tfidf_scores.append(0)

tfidf_map3 = sum(tfidf_scores)/len(tfidf_scores)

print("TF-IDF MAP@3 =", round(tfidf_map3,5))


# MiniLM PIPELINE


model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

minilm_top3 = []
minilm_scores = []

for _, row in train.iterrows():

    prompt_emb = model.encode(row["prompt"], convert_to_tensor=True)

    sims = []

    for opt in options:

        option_emb = model.encode(row[opt], convert_to_tensor=True)

        score = util.cos_sim(prompt_emb, option_emb).item()

        sims.append((opt, score))

    sims.sort(key=lambda x: x[1], reverse=True)

    preds = [x[0] for x in sims[:3]]

    minilm_top3.append(preds)

    if row["answer"] in preds:
        minilm_scores.append(1/(preds.index(row["answer"])+1))
    else:
        minilm_scores.append(0)

minilm_map3 = sum(minilm_scores)/len(minilm_scores)

print("MiniLM MAP@3 =", round(minilm_map3,5))


# COUNT IMPROVEMENTS

count= 0

for i in range(len(train)):

    ans = train.loc[i, "answer"]

    if ans not in tfidf_top3[i] and ans in minilm_top3[i]:
        count += 1

print("Improved Count =", count)

TF-IDF MAP@3 = 0.29617


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


MiniLM MAP@3 = 0.42308
Improved Count = 502


**Zero-shot classification concepts**

QUESTION_11:Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [15]:
import pandas as pd
from transformers import pipeline

# Load dataset
train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

# Initialize zero-shot classification pipeline
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Prompt from row index 1
prompt = train.loc[1, "prompt"]

# Candidate labels = Options A, B, C
candidate_labels = [
    train.loc[1, "A"],
    train.loc[1, "B"],
    train.loc[1, "C"]
]

# Run classifier
result = classifier(
    prompt,
    candidate_labels=candidate_labels
)

print("Labels:", result["labels"])
print("Scores:", result["scores"])

print("\nTop-ranked option:", result["labels"][0])
print("Probability:", round(result["scores"][0], 4))

2026-06-29 09:23:37.916910: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782725018.255832     250 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782725018.342164     250 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782725019.057534     250 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782725019.057572     250 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782725019.057575     250 computation_placer.cc:177] computation placer alr

Labels: ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to impl

QUESTION_12:Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [16]:
import pandas as pd
from transformers import pipeline



# Initialize zero-shot classifier
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Prompt from row index 1
prompt = train.loc[1, "prompt"]

# Candidate labels = Options A, B, C
candidate_labels = [
    train.loc[1, "A"],
    train.loc[1, "B"],
    train.loc[1, "C"]
]


# Softmax (default)

softmax_result = classifier(
    prompt,
    candidate_labels=candidate_labels
)

softmax_sum = sum(softmax_result["scores"])

print("Softmax Scores:", softmax_result["scores"])
print("Softmax Sum:", softmax_sum)


# Independent Sigmoids

multilabel_result = classifier(
    prompt,
    candidate_labels=candidate_labels,
    multi_label=True
)

multilabel_sum = sum(multilabel_result["scores"])

print("\nMulti-label Scores:", multilabel_result["scores"])
print("Multi-label Sum:", multilabel_sum)


# Final Answer

difference = abs(softmax_sum - multilabel_sum)

print("\nAbsolute Difference =", round(difference, 4))

Softmax Scores: [0.4574527442455292, 0.2750643789768219, 0.26748284697532654]
Softmax Sum: 0.9999999701976776

Multi-label Scores: [0.00046927088988013566, 2.0635825421777554e-05, 1.970058110600803e-05]
Multi-label Sum: 0.0005096072964079212

Absolute Difference = 0.9995


QUESTION_13:Let's try Generative AI instead of Classification. 

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 
*


In [17]:
!pip uninstall -y transformers
!pip install transformers==4.40.2 sentence-transformers==2.7.0

Found existing installation: transformers 4.40.2
Uninstalling transformers-4.40.2:
  Successfully uninstalled transformers-4.40.2
  Using cached transformers-4.40.2-py3-none-any.whl.metadata (137 kB)
Using cached transformers-4.40.2-py3-none-any.whl (9.0 MB)


In [18]:
import transformers
print(transformers.__version__)

4.40.2


In [19]:
!pip install -q transformers==4.40.2 sentence-transformers==2.7.0

In [20]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [21]:
import pandas as pd
from transformers import pipeline

# Load dataset
train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

# Load FLAN-T5 Small
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

# Construct the exact prompt
text = (
    f"Question: {train.loc[0, 'prompt']}. "
    f"Is the correct answer A: {train.loc[0, 'A']} "
    f"or B: {train.loc[0, 'B']}? "
    f"Answer with just the letter A or B."
)

print("Prompt:\n")
print(text)

# Generate
result = generator(
    text,
    max_new_tokens=5
)

print("\nModel Output:")
print(result)
print("\nGenerated Text:")
print(result[0]["generated_text"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Prompt:

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.

Model Output:
[{'generated_text': 'B'}]

Generated Text:
B
